In [ ]:
import httpx
import os
from dotenv import load_dotenv

load_dotenv()

FILEVINE_PAT = os.getenv("FILEVINE_PAT")
FILEVINE_CLIENT_ID = os.getenv("FILEVINE_CLIENT_ID")
FILEVINE_CLIENT_SECRET = os.getenv("FILEVINE_CLIENT_SECRET")

print(FILEVINE_PAT is not None, FILEVINE_CLIENT_ID is not None, FILEVINE_CLIENT_SECRET is not None)

True True True


In [27]:
async def get_bearer_token():
    url = "https://identity.filevine.com/connect/token"
    data = {
        "token": FILEVINE_PAT,
        "grant_type": "personal_access_token",
        "scope": "fv.api.gateway.access tenant filevine.v2.api.* openid email fv.auth.tenant.read fv.identity.user.write",
        "client_id": FILEVINE_CLIENT_ID,
        "client_secret": FILEVINE_CLIENT_SECRET,
    }
    headers = {"Content-Type": "application/x-www-form-urlencoded"}

    async with httpx.AsyncClient() as client:
        response = await client.post(url, data=data, headers=headers)
        response.raise_for_status()
        result = response.json()
        return result["access_token"], result["expires_in"]

In [29]:
token, expires_in = await get_bearer_token()
print("Token:", token)
print("Expires in:", expires_in, "seconds")

Token: eyJhbGciOiJSUzUxMiIsImtpZCI6Ijk5MDg4NENGRTRCRTgyQzU4QzA0MDJBQjc5MkE2Rjc5NkI5MUQ2MTBSUzUxMiIsInR5cCI6ImF0K2p3dCIsIng1dCI6Im1RaUV6LVMtZ3NXTUJBS3JlU3B2ZVd1UjFoQSJ9.eyJuYmYiOjE3OTAxNDUwMjIsImV4cCI6MTc5MDE0NjgyMiwiaXNzIjoiaHR0cHM6Ly9pZGVudGl0eS5maWxldmluZS5jb20iLCJhdWQiOlsiZmlsZXZpbmUudjIuYXBpIiwiZnYuYXBpLmdhdGV3YXkiLCJmdi5hdXRoIiwiZnYuaWQiXSwiY2xpZW50X2lkIjoiNEYxODczOEMtMTA3QS00QjgyLUJGQUMtMzA4RjFCNkE2MjZBIiwic3ViIjoiNmViY2U1ODgtZTE2Yy00MTBmLWFkM2UtYmQ5NGI3ZDMxNDU3IiwiYXV0aF90aW1lIjoxNzkwMTQ1MDIyLCJpZHAiOiJsb2NhbCIsInBhdF9pZCI6InQ1ZWF0d0dFWnJvclZMaFArOUYrWFNBQStvUFlDaG4yTTdqc3I0cjZqWXc9IiwicGF0X25hbWUiOiJ6YXBpZXJfdGVzdCIsInBhdF92ZXJzaW9uIjoiMSIsInRlbmFudF9mcm4iOiJmcm46ZmlsZXZpbmU6dXMtcHJvZDpmaWxldmluZS1hcHA6Ojp0ZW5hbnRcXDJjMWU3NTYxLWMxYzEtNGY0OS04MzdlLTAxOWZhZmU2YjY5MSIsInRlbmFudF9pZCI6IjJjMWU3NTYxLWMxYzEtNGY0OS04MzdlLTAxOWZhZmU2YjY5MSIsImp0aSI6IjlFQzcxQjAxRTcwQjFCQThCOEJCMjgwRUI1RjA5QUEyIiwiaWF0IjoxNzkwMTQ1MDIyLCJzY29wZSI6WyJlbWFpbCIsImZpbGV2aW5lLnYyLmFwaS4qIiwiZnYuYXBpLmdhdGV3YXku

In [21]:
async def get_org_id(bearer_token: str):
    url = "https://api.filevineapp.com/fv-app/v2/utils/GetUserOrgsWithToken"
    headers = {"Authorization": f"Bearer {bearer_token}"}

    async with httpx.AsyncClient() as client:
        response = await client.post(url, headers=headers)
        response.raise_for_status()
        return response.json()

In [22]:
org_data = await get_org_id(token)
print(org_data)

HTTPStatusError: Client error '429 Too Many Requests' for url 'https://api.filevineapp.com/fv-app/v2/utils/GetUserOrgsWithToken'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/429

In [30]:
#org_id = org_data["orgId"]
org_id = "28"

headers = {
    "Authorization": f"Bearer {token}",
    "x-fv-orgid": org_id,
}

async with httpx.AsyncClient() as client:
    response = await client.get("https://api.filevineapp.com/fv-app/v2/projects", headers=headers)
    response.raise_for_status()
    print(response.json())

HTTPStatusError: Client error '429 Too Many Requests' for url 'https://api.filevineapp.com/fv-app/v2/projects'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/429

In [31]:
async with httpx.AsyncClient() as client:
    response = await client.get("https://api.filevineapp.com/fv-app/v2/projects", headers=headers)
    print("Status:", response.status_code)
    print("Headers:", dict(response.headers))
    print("Body:", response.text)

Status: 429
Headers: {'date': 'Wed, 23 Sep 2026 06:39:08 GMT', 'content-length': '0', 'connection': 'keep-alive', 'server': 'cloudflare', 'ratelimit-limit': '1;r=5.9999999999999995E-05;w=60;c=Customer Default (legacy - superseded)', 'ratelimit-remaining': '0;r=5.9999999999999995E-05;w=60;c=Customer Default (legacy - superseded)', 'x-fv-correlation-id': 'a89556a19c9a40958dd2bf6f40d7908d', 'cf-cache-status': 'DYNAMIC', 'cf-ray': 'a3f78ce79a334731-PMO', 'alt-svc': 'h3=":443"; ma=86400'}
Body: 
